In [0]:
from pyspark.sql import functions as F

silver = spark.read.table("workspace.taxi_lakehouse.silver_trips")

gold_daily = (
    silver.groupBy("pickup_date")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("trip_duration_min"), 2).alias("avg_duration_min")
    )
    .orderBy("pickup_date")
)
gold_daily.display()

pickup_date,total_trips,total_revenue,avg_fare,avg_distance,avg_duration_min
2016-01-01,326,3998.0,12.26,3.13,20.14
2016-01-02,337,4022.0,11.93,3.01,28.24
2016-01-03,275,3403.0,12.37,3.19,11.06
2016-01-04,322,3890.5,12.08,3.0,11.56
2016-01-05,357,4054.0,11.36,2.61,19.9
2016-01-06,358,4133.0,11.54,2.63,12.4
2016-01-07,353,4294.0,12.16,2.94,16.97
2016-01-08,390,4934.0,12.65,2.94,17.25
2016-01-09,410,4858.0,11.85,2.85,15.75
2016-01-10,345,4050.5,11.74,3.03,15.1


In [0]:
gold_hourly = (
    silver.groupBy("pickup_hour")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare")
    )
    .orderBy("pickup_hour")
)
gold_hourly.display()

pickup_hour,total_trips,avg_fare
0,741,13.22
1,570,12.8
2,443,13.05
3,304,13.37
4,247,14.89
5,218,14.78
6,474,12.14
7,798,11.58
8,1033,12.02
9,1038,12.01


In [0]:
gold_top_zones = (
    silver.groupBy("pickup_zip")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue")
    )
    .orderBy(F.desc("total_trips"))
    .limit(15)
)
gold_top_zones.display()

pickup_zip,total_trips,total_revenue
10001,1227,13028.01
10003,1180,12963.0
10011,1128,12319.0
10021,1017,10321.5
10018,1010,11539.01
10023,1006,10112.0
10028,927,9432.5
10012,831,9456.0
10110,761,8261.5
10065,700,6837.5


In [0]:
gold_daily.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_lakehouse.gold_daily_summary")
gold_hourly.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_lakehouse.gold_hourly_demand")
gold_top_zones.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_lakehouse.gold_top_pickup_zones")
print("All 3 gold tables created ✅")

All 3 gold tables created ✅
